In [15]:
import GIN
from torchinfo import summary
import torch

In [16]:
gin = GIN.GIN()

In [4]:
print(gin)

GIN(
  (node_encoder): Linear(in_features=26, out_features=256, bias=True)
  (edge_encoder): Linear(in_features=6, out_features=256, bias=True)
  (virtual_node_embedding): Embedding(1, 256)
  (convs): ModuleList(
    (0-6): 7 x GINEConv(nn=Sequential(
      (0): Linear(in_features=256, out_features=256, bias=True)
      (1): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU()
      (3): Linear(in_features=256, out_features=256, bias=True)
    ))
  )
  (batch_norms): ModuleList(
    (0-6): 7 x BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  )
  (vn_mlp): ModuleList(
    (0-6): 7 x Sequential(
      (0): Linear(in_features=256, out_features=256, bias=True)
      (1): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU()
      (3): Linear(in_features=256, out_features=256, bias=True)
      (4): ReLU()
      (5): Dropout(p=0.05, inplace=False)
    )
  )
  (predicti

In [8]:
summary(gin)

Layer (type:depth-idx)                   Param #
GIN                                      --
├─Linear: 1-1                            6,912
├─Linear: 1-2                            1,792
├─Embedding: 1-3                         256
├─ModuleList: 1-4                        --
│    └─GINEConv: 2-1                     --
│    │    └─SumAggregation: 3-1          --
│    │    └─Sequential: 3-2              132,096
│    └─GINEConv: 2-2                     --
│    │    └─SumAggregation: 3-3          --
│    │    └─Sequential: 3-4              132,096
│    └─GINEConv: 2-3                     --
│    │    └─SumAggregation: 3-5          --
│    │    └─Sequential: 3-6              132,096
│    └─GINEConv: 2-4                     --
│    │    └─SumAggregation: 3-7          --
│    │    └─Sequential: 3-8              132,096
│    └─GINEConv: 2-5                     --
│    │    └─SumAggregation: 3-9          --
│    │    └─Sequential: 3-10             132,096
│    └─GINEConv: 2-6                   

In [17]:
# High-quality architecture visualization setup (torchview + Graphviz).
import sys
import subprocess
from torch_geometric.data import Data

try:
    from torchview import draw_graph
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'torchview', 'graphviz'])
    from torchview import draw_graph


[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [20]:
class GINTraceWrapper(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, x, edge_index, edge_attr, batch):
        data = Data(x=x, edge_index=edge_index, edge_attr=edge_attr, batch=batch)
        return self.model(data)


gin.eval()
wrapped = GINTraceWrapper(gin)

# Dummy single-graph input matching GIN defaults: node_in_dim=26, edge_in_dim=6
num_nodes = 8
x = torch.randn(num_nodes, 26)
edge_index = torch.tensor(
    [[0, 1, 1, 2, 2, 3, 3, 4, 4, 5, 5, 6, 6, 7, 1, 0, 2, 1, 3, 2, 4, 3, 5, 4, 6, 5, 7, 6],
     [1, 0, 2, 1, 3, 2, 4, 3, 5, 4, 6, 5, 7, 6, 0, 1, 1, 2, 2, 3, 3, 4, 4, 5, 5, 6, 6, 7]],
    dtype=torch.long,
 )
edge_attr = torch.randn(edge_index.size(1), 6)
batch = torch.zeros(num_nodes, dtype=torch.long)

try:
    model_graph = draw_graph(
        wrapped,
        input_data=(x, edge_index, edge_attr, batch),
        graph_name='GIN Architecture',
        depth=4,
        expand_nested=True,
        roll=False,
        save_graph=True,
        filename='gin_architecture',
        directory='runs/architecture',
    )
    model_graph.visual_graph
    print('Saved architecture graph to runs/architecture/gin_architecture.*')
except Exception as exc:
    print(f'torchview rendering failed: {exc}')
    print('Falling back to TensorBoard graph writer...')
    from torch.utils.tensorboard import SummaryWriter
    writer = SummaryWriter('runs/experiment_1')
    writer.add_graph(wrapped, (x, edge_index, edge_attr, batch))
    writer.close()
    print('Graph written to runs/experiment_1. Launch with: tensorboard --logdir runs')

Saved architecture graph to runs/architecture/gin_architecture.*
